In [0]:
from pyspark.sql import functions as F, DataFrame
from datetime import datetime

# ── Configuração ──────────────────────────────────────────────────────────────
SOURCE_TABLE = "homologacao.salutar_saude.raw_contratos"
TARGET_TABLE = "homologacao.salutar_saude.silver_contratos"

DATE_COLS = ["data_venda", "vigencia_inicio", "vigencia_fim"]

run_ts = datetime.now()

print(f"Pipeline  : silver_contratos")
print(f"Iniciado  : {run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Origem    : {SOURCE_TABLE}")
print(f"Destino   : {TARGET_TABLE}")

Pipeline  : silver_contratos
Iniciado  : 2026-07-04 20:51:29
Origem    : homologacao.salutar_saude.raw_contratos
Destino   : homologacao.salutar_saude.silver_contratos


In [0]:
df_raw = spark.table(SOURCE_TABLE)
n_raw = df_raw.count()

print(f"[OK] {n_raw:,} registros lidos de {SOURCE_TABLE}")

[OK] 109 registros lidos de homologacao.salutar_saude.raw_contratos


In [0]:
def parse_date(col_name: str) -> F.Column:
    """Normaliza datas para YYYY-MM-DD.
    Suporta os formatos: YYYY-MM-DD, DD-MM-YYYY, DD/MM/YYYY.
    Usa try_to_date para retornar NULL em vez de lançar exceção.
    """
    return F.date_format(
        F.coalesce(
            F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd-MM-yyyy')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd/MM/yyyy')"),
        ),
        "yyyy-MM-dd",
    ).alias(col_name)


def parse_valor_mensal(col_name: str) -> F.Column:
    """Normaliza valor_mensal para decimal(15,2).
    Trata três formatos encontrados na fonte:
      - 'R$ 130.317,94'  → formato brasileiro com prefixo  (. milhar, , decimal)
      - '140540,13'       → sem prefixo, vírgula decimal
      - '198111.30'       → sem prefixo, ponto decimal
    """
    c = F.trim(F.regexp_replace(F.col(col_name), r"R\$\s*", ""))  # remove 'R$ '
    # Formato brasileiro: tem vírgula → remove ponto (milhar), troca vírgula por ponto
    br_fmt = F.regexp_replace(F.regexp_replace(c, r"\.", ""), ",", ".")
    # Formato inglês: sem vírgula → ponto já é decimal, usa direto
    return (
        F.when(F.col(col_name).contains(","), br_fmt)
         .otherwise(c)
         .cast("decimal(15,2)")
         .alias(col_name)
    )


def parse_status(col_name: str) -> F.Column:
    """Normaliza status: remove espaços extras e padroniza capitalização.
    Exemplos: 'Ativo ' → 'Ativo' | 'ATIVO' → 'Ativo' | 'ativo' → 'Ativo'
    """
    return F.initcap(F.trim(F.col(col_name))).alias(col_name)


def transform(df: DataFrame, cols: list) -> DataFrame:
    """Aplica todas as transformações de padronização da camada silver.
    Remove a coluna _rescued_data (artefato da camada RAW) e
    adiciona _updated_at com o timestamp de execução.

    Args:
        df  : DataFrame de origem (camada RAW).
        cols: lista de colunas pré-computada fora da função (evita RPC repetido
              ao acessar df.columns dentro de um loop/função no Spark Connect).
    """
    return df.select(
        *[
            parse_date(c)         if c in DATE_COLS       else
            parse_valor_mensal(c) if c == "valor_mensal"  else
            parse_status(c)       if c == "status"        else
            F.col(c)
            for c in cols
        ],
        F.lit(run_ts).cast("timestamp").alias("_updated_at"),  # evita .withColumn em cadeia
    )


print("[OK] Funções de transformação definidas.")

[OK] Funções de transformação definidas.


In [0]:
# Computa schema uma única vez fora da função (evita RPC repetido no Spark Connect)
silver_cols = [c for c in df_raw.columns if c != "_rescued_data"]
df_silver = transform(df_raw, silver_cols)

# ── Validação de qualidade ──────────────────────────────────────────────────────────────
DQ_COLS = DATE_COLS + ["valor_mensal", "status"]

null_counts = (
    df_silver
    .select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in DQ_COLS])
    .collect()[0]
    .asDict()
)
unexpected_status = df_silver.filter(
    F.col("status").isNotNull() & ~F.col("status").isin("Ativo", "Cancelado", "Renovado")
).count()
n_silver = df_silver.count()

print("── Data Quality ───────────────────────────────────────────────────────────")
print(f"  Total de registros             : {n_silver:,}")
print(f"  Correspondência com RAW        : {'[OK]' if n_silver == n_raw else '[WARN] divergência!'}")
for col_name, nulls in null_counts.items():
    tag = "[WARN]" if nulls > 0 else "[OK]  "
    print(f"  {tag} {col_name}: {nulls:,} nulos")
tag = "[WARN]" if unexpected_status else "[OK]  "
print(f"  {tag} status valores inesperados : {unexpected_status:,}")
print("─" * 65)

assert n_silver == n_raw, f"Contagem divergente: RAW={n_raw}, silver={n_silver}"
assert unexpected_status == 0, f"{unexpected_status} valores inesperados em 'status'"

── Data Quality ───────────────────────────────────────────────────────────
  Total de registros             : 109
  Correspondência com RAW        : [OK]
  [OK]   data_venda: 0 nulos
  [OK]   vigencia_inicio: 0 nulos
  [OK]   vigencia_fim: 0 nulos
  [WARN] valor_mensal: 1 nulos
  [OK]   status: 0 nulos
  [OK]   status valores inesperados : 0
─────────────────────────────────────────────────────────────────


In [0]:
from delta.tables import DeltaTable

# ── Escrita incremental via MERGE ─────────────────────────────────────────────
# Chave de merge: contrato_id (chave primária da tabela)
MERGE_KEY = "contrato_id"

# Remove duplicadas de contrato_id (equivalente ao DISTINCT no SQL)
df_to_merge = df_silver.dropDuplicates([MERGE_KEY])

if spark.catalog.tableExists(TARGET_TABLE):
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_target.alias("target")
        .merge(
            df_to_merge.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        # .whenNotMatchedBySourceDelete()
        .execute()
    )
    print(f"[OK] MERGE concluído      : {TARGET_TABLE}")
else:
    # Primeira execução: cria a tabela (carga inicial)
    df_to_merge.write.format("delta").saveAsTable(TARGET_TABLE)
    print(f"[OK] Carga inicial        : {TARGET_TABLE}")

n_written = spark.table(TARGET_TABLE).count()
print(f"[OK] Registros na tabela  : {n_written:,}")
print(f"[OK] Fim do pipeline      : {datetime.now():%Y-%m-%d %H:%M:%S}")

[OK] Carga inicial        : homologacao.salutar_saude.silver_contratos
[OK] Registros na tabela  : 108
[OK] Fim do pipeline      : 2026-07-04 20:51:36


In [0]:
%sql
SELECT *
FROM homologacao.salutar_saude.silver_contratos

contrato_id,qtd
